## Step 03 - Research Agent With Tool

Goal: add one search tool to the research agent so results are recent and source-backed.

### What's New in This Step

- Step 02 relied only on model memory for research output.
- This step adds a built-in tool (`SerperDevTool`) so the agent can fetch fresher web data.
- The task now asks for source links and dates, introducing a basic quality guardrail.

In [ ]:
import os
from dotenv import load_dotenv
from crewai import LLM, Agent, Crew, Task
from crewai_tools import SerperDevTool

load_dotenv()

TOPIC = "Platform Engineering Best Practices"

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")

if not openrouter_api_key:
    raise ValueError("Missing OPENROUTER_API_KEY")
if not serper_api_key:
    raise ValueError("Missing SERPER_API_KEY")


In [ ]:
# LLM: shared reasoning model used by the agent.
llm = LLM(
    model="openai/gpt-4o",
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)

# Built-in tool: enables live web search instead of memory-only answers.
search_tool = SerperDevTool()

# Agent: research specialist that can call the search tool.
# verbose=True so the audience can see the reason -> tool call -> observation
# loop. Turn it off again in later notebooks once the mechanism is understood.
research_agent = Agent(
    role="Research Analyst",
    goal="Find latest updates on {topic}",
    backstory="You give short, factual, source-backed summaries.",
    llm=llm,
    tools=[search_tool],
    verbose=True,
)


In [ ]:
# Task: requires dated, link-backed findings to improve reliability.
research_task = Task(
    description=(
        "Research {topic} from 2026 onward. "
        "Use the search tool and include source links with dates."
    ),
    expected_output="A short 8 to 10 bullet summary with sources.",
    agent=research_agent,
)

# verbose=True at the Crew level mirrors the agent setting and prints the
# task lifecycle (start, tool call, output) — useful for first-time learners.
crew = Crew(
    agents=[research_agent],
    tasks=[research_task],
    verbose=True,
)


In [ ]:
result = crew.kickoff(inputs={"topic": TOPIC})
print(getattr(result, "raw", str(result)))


### Recap

- LLM did: synthesize search findings into a readable summary.
- Agent did: choose when to call the built-in tool and how to present results.
- Task enforced: freshness and source-link expectations.

